In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass


EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.image-classify.food-quality",
        description="Classifies the food quality of a product in an image into one of the five categories: Pass_ripe, Pass_ok, Fail_contaminated, Fail_unripe, or Fail_defective",
        worker_release="qwen3-instruct",
        text_prompt="""
          Determine the primary content of the image and assign exactly one label:
          ['Pass_ripe', 'Pass_ok', 'Fail_unripe', 'Fail_defective', 'Fail_contaminated'].
          Focus only on the single main food item or package that is the clear subject of the image.
          Ignore background items, shelving, hands, and unrelated objects.
          Choose 'Pass_ripe' only if the image's main focus is a piece of fresh produce at peak ripeness,
          showing full expected color, firmness, and no visible bruising, mold, discoloration, or damage.
          This label applies to produce only.
          Choose 'Pass_ok' only if the image's main focus is a packaged, non-produce food item that is structurally
          intact and sealed, with no tears, dents, leaks, or breaks. This label applies to packaged goods, not produce.
          Choose 'Fail_unripe' only if the image's main focus is a piece of fresh produce that is clearly underripe or
          immature, such as a green banana, hard unripened avocado, or pale, underdeveloped fruit. The produce should
          otherwise be firm and free of damage, mold, or spoilage. Do not choose this label for packaged goods.
          Choose 'Fail_defective' only if the image's main focus shows clear physical or structural damage with no signs
          of spoilage or contamination. This includes a dented or crushed box, a torn or punctured bag with contents still
          fully contained, a cracked jar or bottle lid with no leakage, or a dented can. Do not choose this label if any
          leaking, mold, dirt, pests, or spoilage is visible.
          Choose 'Fail_contaminated' only if the image's main focus shows visible spoilage, mold, bruising on produce,
          leaking contents, dirt or debris, or signs of pest activity such as holes or droppings. This applies to both
          produce and packaged goods whenever contamination or spoilage is visibly present, regardless of cause.
          If multiple issues are visible, choose the most visually dominant and severe issue, prioritizing contamination
          over defectiveness if both are present.
          Return only the single best-fitting label.

        """,
        transform_into=TransformInto(),
       config=InferRuntimeConfig(
           max_new_tokens=5,
           image_size=640
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image


In [ ]:
from pathlib import Path


pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.image-classify.food-quality:latest"
   )
])


with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("/content/sample_img.jpg") # change to the path of your sample image
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
     print(json.dumps(result, indent=2))

print("Done")
